In [1]:
include("../src/Apoblast.jl")
using .Apoblast
using SymPy

In [4]:
model = Model(("x", "y"), ("u_x", "u_y"))

x, y = model.coords
ux, uy = model.fields

(u_x(x, y), u_y(x, y))

In [20]:
monoterms = (
    (ux, 0),
    (uy, 0),
    (diff(ux, x), 1),
    (diff(ux, y), 1),
    (diff(uy, x), 1),
    (diff(uy, y), 1),
    (diff(ux, x, x), 2),
    (diff(ux, x, y), 2),
    (diff(ux, y, y), 2),
    (diff(uy, x, x), 2),
    (diff(uy, x, y), 2),
    (diff(uy, y, y), 2),
)

function listorder(monoterms, order, dorder, order_max, dorder_max; filter = (o,d)->true)
    if order_max < order || dorder_max < dorder
        return []
    elseif isempty(monoterms)
        if filter(order, dorder)
            return [Sym(1)]
        else
            return []
        end
    else
        arr = Sym[]
        var, weight = monoterms[1]

        for i in 0:(order_max - order)
            sub = listorder(
                monoterms[2:end],
                order + i,
                dorder + i * weight,
                order_max,
                dorder_max;
                filter=filter
            )
            for t in sub
                push!(arr, (var^i) * t)
            end
        end

        return arr
    end
end

d = 2
zet = 2
chi = 0

filter_func = (o, d) -> o*chi - d >= chi - 2

list = listorder(monoterms, 0, 0, 3, 2; filter=filter_func)
lib = Library(model, Tuple(list))

Library((1, Derivative(u_y(x, y), (y, 2)), Derivative(u_y(x, y), x, y), Derivative(u_y(x, y), (x, 2)), Derivative(u_x(x, y), (y, 2)), Derivative(u_x(x, y), x, y), Derivative(u_x(x, y), (x, 2)), Derivative(u_y(x, y), y), Derivative(u_y(x, y), y)^2, Derivative(u_y(x, y), x), Derivative(u_y(x, y), x)*Derivative(u_y(x, y), y), Derivative(u_y(x, y), x)^2, Derivative(u_x(x, y), y), Derivative(u_x(x, y), y)*Derivative(u_y(x, y), y), Derivative(u_x(x, y), y)*Derivative(u_y(x, y), x), Derivative(u_x(x, y), y)^2, Derivative(u_x(x, y), x), Derivative(u_x(x, y), x)*Derivative(u_y(x, y), y), Derivative(u_x(x, y), x)*Derivative(u_y(x, y), x), Derivative(u_x(x, y), x)*Derivative(u_x(x, y), y), Derivative(u_x(x, y), x)^2, u_y(x, y), u_y(x, y)*Derivative(u_y(x, y), (y, 2)), u_y(x, y)*Derivative(u_y(x, y), x, y), u_y(x, y)*Derivative(u_y(x, y), (x, 2)), u_y(x, y)*Derivative(u_x(x, y), (y, 2)), u_y(x, y)*Derivative(u_x(x, y), x, y), u_y(x, y)*Derivative(u_x(x, y), (x, 2)), u_y(x, y)*Derivative(u_y(x, y),

In [21]:
trans_1 = Transformation(model,
    (-x, y),
    (-ux, uy);
)

θ = Sym("θ")
trans_2 = Transformation(model,
    (x*cos(θ) + y*sin(θ), y*cos(θ) - x*sin(θ)),
    (ux*cos(θ) + uy*sin(θ), uy*cos(θ) - ux*sin(θ));
    parameter = ((θ, Sym(0)),)
)

Transformation((x*cos(θ) + y*sin(θ), -x*sin(θ) + y*cos(θ)), (u_x(x, y)*cos(θ) + u_y(x, y)*sin(θ), -u_x(x, y)*sin(θ) + u_y(x, y)*cos(θ)), Dict{Sym, Sym}(x => x*cos(θ) + y*sin(θ), y => -x*sin(θ) + y*cos(θ)), Dict{Sym, Sym}(u_x(x, y) => u_x(x, y)*cos(θ) + u_y(x, y)*sin(θ), u_y(x, y) => -u_x(x, y)*sin(θ) + u_y(x, y)*cos(θ)), (θ,), Dict{Sym, Sym}(θ => 0), Sym[-sin(θ)^2/cos(θ) + 1/cos(θ) -sin(θ); sin(θ) cos(θ)])

In [22]:
Li = [ux, uy]
f̃ = [ux, uy]

result = Apoblast.Core.collect_follower(model, lib, Li, f̃, trans_1, trans_2)

for list in result
    display(list)
end

kernel dim : 200
library dim: 100
stage 1 / 2 : Transformation((-x, y), (-u_x(x, y), u_y(x, y)), Dict{Sym, Sym}(x => -x, y => y), Dict{Sym, Sym}(u_x(x, y) => -u_x(x, y), u_y(x, y) => u_y(x, y)), (), Dict{Sym, Sym}(), Sym[-1 0; 0 1])
  constraint matrix size: (2, 2)
  leak terms: Sym[]
  constraint matrix size: (200, 200)
  nullity of constraint matrix: 100
stage 2 / 2 : Transformation((x*cos(θ) + y*sin(θ), -x*sin(θ) + y*cos(θ)), (u_x(x, y)*cos(θ) + u_y(x, y)*sin(θ), -u_x(x, y)*sin(θ) + u_y(x, y)*cos(θ)), Dict{Sym, Sym}(x => x*cos(θ) + y*sin(θ), y => -x*sin(θ) + y*cos(θ)), Dict{Sym, Sym}(u_x(x, y) => u_x(x, y)*cos(θ) + u_y(x, y)*sin(θ), u_y(x, y) => -u_x(x, y)*sin(θ) + u_y(x, y)*cos(θ)), (θ,), Dict{Sym, Sym}(θ => 0), Sym[-sin(θ)^2/cos(θ) + 1/cos(θ) -sin(θ); sin(θ) cos(θ)])
  constraint matrix size: (2, 2)
  leak terms: Sym[]
  constraint matrix size: (200, 200)
  nullity of constraint matrix: 18


2-element Vector{Sym}:
 Derivative(u_x(x, y), (x, 2)) + Derivative(u_x(x, y), (y, 2))
 Derivative(u_y(x, y), (x, 2)) + Derivative(u_y(x, y), (y, 2))

2-element Vector{Sym}:
 -Derivative(u_x(x, y), (x, 2)) + Derivative(u_y(x, y), x, y)
 -Derivative(u_y(x, y), (y, 2)) + Derivative(u_x(x, y), x, y)

2-element Vector{Sym}:
 uₓ(x, y)
 u_y(x, y)

2-element Vector{Sym}:
 u_x(x, y)*Derivative(u_x(x, y), y)^2 + 2*u_x(x, y)*Derivative(u_x(x, y), y)*Derivative(u_y(x, y), x) + u_x(x, y)*Derivative(u_y(x, y), x)^2
 u_y(x, y)*Derivative(u_x(x, y), y)^2 + 2*u_y(x, y)*Derivative(u_x(x, y), y)*Derivative(u_y(x, y), x) + u_y(x, y)*Derivative(u_y(x, y), x)^2

...

2-element Vector{Sym}:
 u_x(x, y)^2*Derivative(u_x(x, y), (x, 2)) - u_x(x, y)^2*Derivative(u_y(x, y), x, y) + u_x(x, y)*u_y(x, y)*Derivative(u_y(x, y), (y, 2)) - u_x(x, y)*u_y(x, y)*Derivative(u_x(x, y), x, y)
 u_x(x, y)*u_y(x, y)*Derivative(u_x(x, y), (x, 2)) - u_x(x, y)*u_y(x, y)*Derivative(u_y(x, y), x, y) + u_y(x, y)^2*Derivative(u_y(x, y), (y, 2)) - u_y(x, y)^2*Derivative(u_x(x, y), x, y)

2-element Vector{Sym}:
 u_x(x, y)^2*Derivative(u_x(x, y), (y, 2)) + 2*u_x(x, y)*u_y(x, y)*Derivative(u_x(x, y), x, y) + u_y(x, y)^2*Derivative(u_x(x, y), (x, 2))
 u_x(x, y)^2*Derivative(u_y(x, y), (y, 2)) + 2*u_x(x, y)*u_y(x, y)*Derivative(u_y(x, y), x, y) + u_y(x, y)^2*Derivative(u_y(x, y), (x, 2))

2-element Vector{Sym}:
 u_x(x, y)^2*Derivative(u_x(x, y), (x, 2)) - 2*u_x(x, y)*u_y(x, y)*Derivative(u_x(x, y), x, y) + u_y(x, y)^2*Derivative(u_x(x, y), (y, 2))
 u_x(x, y)^2*Derivative(u_y(x, y), (x, 2)) - 2*u_x(x, y)*u_y(x, y)*Derivative(u_y(x, y), x, y) + u_y(x, y)^2*Derivative(u_y(x, y), (y, 2))

2-element Vector{Sym}:
 -u_x(x, y)^2*Derivative(u_x(x, y), (x, 2)) + u_x(x, y)^2*Derivative(u_x(x, y), (y, 2)) + u_x(x, y)^2*Derivative(u_y(x, y), x, y) + 2*u_x(x, y)*u_y(x, y)*Derivative(u_x(x, y), x, y) + u_y(x, y)^2*Derivative(u_y(x, y), x, y)
  u_x(x, y)^2*Derivative(u_x(x, y), x, y) + 2*u_x(x, y)*u_y(x, y)*Derivative(u_y(x, y), x, y) + u_y(x, y)^2*Derivative(u_y(x, y), (x, 2)) - u_y(x, y)^2*Derivative(u_y(x, y), (y, 2)) + u_y(x, y)^2*Derivative(u_x(x, y), x, y)

2-element Vector{Sym}:
 u_x(x, y)^3 + u_x(x, y)*u_y(x, y)^2
 u_x(x, y)^2*u_y(x, y) + u_y(x, y)^3